In [1]:
!pip install soundfile

In [2]:
!git clone https://github.com/clovaai/aasist.git /kaggle/working/aasist

Cloning into '/kaggle/working/aasist'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 38 (delta 12), reused 6 (delta 6), pack-reused 10 (from 1)
Receiving objects: 100% (38/38), 1.43 MiB | 23.95 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [3]:
# 新版 PyTorch 已内置 SWA，不需要 torchcontrib
# 修改 main.py 中的 import
import fileinput, sys
# 替换 main.py 中的 torchcontrib
with open('/kaggle/working/aasist/main.py', 'r') as f:
    content = f.read()

content = content.replace(
    'from torchcontrib.optim import SWA',
    'from torch.optim.swa_utils import AveragedModel, SWALR, update_bn'
)
# SWA 用法也要改
content = content.replace(
    'optimizer_swa = SWA(optimizer)',
    'swa_model = AveragedModel(model)\nswa_scheduler = SWALR(optimizer, swa_lr=optim_config["base_lr"])'
)
content = content.replace(
    'optimizer_swa.update_swa()',
    'swa_model.update_parameters(model)'
)
content = content.replace(
    'optimizer_swa.swap_swa_sgd()',
    '# SWA: copy averaged params to model'
)
content = content.replace(
    'optimizer_swa.bn_update(trn_loader, model, device=device)',
    'update_bn(trn_loader, swa_model, device=device)\nmodel.load_state_dict(swa_model.module.state_dict())'
)

with open('/kaggle/working/aasist/main.py', 'w') as f:
    f.write(content)
print("main.py patched for new PyTorch SWA API")

main.py patched for new PyTorch SWA API


In [4]:
!rm -rf /kaggle/working/aasist/LA

In [5]:
!ln -s /kaggle/input/datasets/anishsarkar22/asvpoof-2019-dataset-la/LA /kaggle/working/aasist/LA
!ls /kaggle/working/aasist/LA/

ASVspoof2019_LA_asv_protocols  ASVspoof2019_LA_dev    README.LA.txt
ASVspoof2019_LA_asv_scores     ASVspoof2019_LA_eval
ASVspoof2019_LA_cm_protocols   ASVspoof2019_LA_train


In [6]:
# 可选：减小 batch_size
import json
with open('/kaggle/working/aasist/config/AASIST.conf', 'r') as f:
    config = json.load(f)
config['batch_size'] = 16
with open('/kaggle/working/aasist/config/AASIST.conf', 'w') as f:
    json.dump(config, f, indent=4)

In [7]:
%cd /kaggle/working/aasist
!python main.py --config /kaggle/working/aasist/config/AASIST.conf --output_dir /kaggle/working/exp_result

/kaggle/working/aasist
  File "/kaggle/working/aasist/main.py", line 118
    best_dev_eer = 1.
IndentationError: unexpected indent
